# Part A: Simple (Non-Optimized) Collection Plan
AIPI 530 · Fall 2026 · Oklahoma Mothers' Milk Bank

**Algorithm: cluster first, route second.** Group the depots into K regions around centers chosen by driving time,
run one route per region (farthest depot first, then nearest neighbor, always checking that the truck can still
get back to the bank within 720 minutes), send leftover depots on extra routes on the unused weekdays, and ship
whatever is still unserved. No optimization model,solver, or routing library used.

**Packages:** `pandas`, `openpyxl`. **Input files** (`data` folder): `Depot Information.xlsx`,
`Driving Distance and Time.xlsx`. Open this notebook from the `code` folder and run the cells top to bottom.

### Load packages

In [6]:
import pandas as pd

### Load data

In [7]:
depotData = pd.read_excel('../data/Depot Information.xlsx', sheet_name='Depots')
travelData = pd.read_excel('../data/Optional - Driving Distance and Time.xlsx')
# one row for every ordered pair (from, to); Time is in minutes and Distance in miles
travelData.columns = ['Pair', 'From', 'To', 'Time', 'Distance']

BANK = 'Oklahoma Mothers’ Milk Bank'
DEPOTS = [d for d in depotData['Depot Designation'] if d != BANK]   # the 36 depots, in workbook order
# dicts to look up time from i to j and distance from i to j
time = travelData.set_index(['From', 'To'])['Time'].to_dict()        # time[i, j] can differ from time[j, i]
dist = travelData.set_index(['From', 'To'])['Distance'].to_dict()

print(len(DEPOTS), 'depots,', len(time), 'directed driving-time pairs')

36 depots, 1332 directed driving-time pairs


### Case parameters (used unchanged)

In [8]:
FUEL_PRICE = 2.70        # $ per gallon
MPG = 21                 # miles per gallon
LABOR_RATE = 7.25        # $ per driving hour
SHIP_RATE = 1.00         # $ per mile, depot to bank
MAX_MINUTES = 720        # maximum driving time per route, including the return to the bank
MAX_ROUTES = 5           # Monday to Friday

laborPerMin = LABOR_RATE / 60
fuelPerMile = FUEL_PRICE / MPG

### The algorithm
**Inputs:** the 36 depots; directed driving times and distances; the 720-minute and 5-route limits.
**Outputs:** ordered stops for each daily route, and the list of depots that ship directly.

1. **Closeness.** The closeness of two depots is the driving time there plus the driving time back (the data is directed, so this gives one number).
2. **Choose K.** For K = 2, 3, 4, 5 build the plan below. Keep the largest K for which no depot has to ship. (On this data: K = 3.)
3. **Pick K centers.** The first center is the depot farthest from the bank. Each next center is the depot farthest from all centers chosen so far.
4. **Assign.** Every depot joins the cluster of its nearest center.
5. **Route each cluster (one route per cluster).** Start at the bank. First stop: the cluster's depot that is farthest from the bank.
   Next stops: the nearest unserved depot in the cluster, as long as (minutes so far + drive there + drive from there back to the bank) is at most 720.
   When nothing fits, return to the bank.
6. **Repair.** Pool the depots that did not fit. While weekdays remain, run the same routing rule (step 5) on the pool as a new route.
7. **Ship.** Any depot still unserved after five routes ships directly.
8. **Days.** Routes are labeled Monday, Tuesday, ... in the order they were built.

Ties are broken by workbook order, so the result is reproducible.

In [9]:
def closeness(i, j):
    return 0 if i == j else time[i, j] + time[j, i] # time there and back added up

def pick_centers(K):
    centers = [max(DEPOTS, key=lambda j: closeness(BANK, j))] # farthest depot from the bank
    # add other farthest centers from already picked to the list (K centers)
    while len(centers) < K:
        centers.append(max((d for d in DEPOTS if d not in centers),
                           key=lambda d: min(closeness(d, c) for c in centers)))
    return centers

def assign_clusters(K):
    centers = pick_centers(K)
    clusters = {c: [] for c in centers} # empty dict: list for each center
    # every depot joins the cluster of its nearest center
    for d in DEPOTS:
        clusters[min(centers, key=lambda c: closeness(d, c))].append(d)
    return clusters

def build_route(pool):
    """One route over the depots in pool. Returns (route or None, depots that did not fit)."""
    unserved = set(pool)
    route, minutes = [BANK], 0.0
    while True:
        here = route[-1]
        # which unserved depots can I still visit and get back to the bank within 720 minutes?
        fits = [j for j in pool if j in unserved and minutes + time[here, j] + time[j, BANK] <= MAX_MINUTES]
        if not fits:
            break
        if here == BANK:
            nxt = max(fits, key=lambda j: time[BANK, j])      # first stop: depot farthest from the bank
        else:
            nxt = min(fits, key=lambda j: time[here, j])      # then nearest neighbor - nearest depot from where the truck is now
        minutes += time[here, nxt]
        route.append(nxt)
        unserved.remove(nxt)
    if len(route) == 1: # nothing fit
        return None, unserved
    route.append(BANK) # drive back to bank
    return route, unserved

def build_plan(K):
    routes, leftover = [], []
    # one route for each cluster
    for center, members in assign_clusters(K).items():
        route, notServed = build_route(members)
        if route:
            routes.append(route)
        leftover += [d for d in members if d in notServed]
    firstLeftover = len(leftover)
    # repair step: extra routes for the leftover depots while there are weekdays left
    pool = leftover
    while pool and len(routes) < MAX_ROUTES:
        route, notServed = build_route(pool)
        if not route:
            break
        routes.append(route)
        pool = [d for d in pool if d in notServed]
    return routes, pool, firstLeftover  # whatever is left in pool ships directly


def cost_of(routes, ship):
    T = sum(time[a, b] for r in routes for a, b in zip(r[:-1], r[1:])) # total min
    D = sum(dist[a, b] for r in routes for a, b in zip(r[:-1], r[1:])) # total miles
    S = sum(SHIP_RATE * dist[i, BANK] for i in ship) # total shipping cost
    return T, D, laborPerMin * T, fuelPerMile * D, S, laborPerMin * T + fuelPerMile * D + S

### Step 2: choose K

In [10]:
# try K = 2, 3, 4, 5
rows = []
for K in [2, 3, 4, 5]:
    routes_k, ship_k, left_k = build_plan(K)
    rows.append({'K': K, 'Cluster sizes': sorted(len(m) for m in assign_clusters(K).values()),
                 'Depots not on cluster routes': left_k, 'Routes': len(routes_k),
                 'Depots shipped': len(ship_k), 'Total cost $': cost_of(routes_k, ship_k)[5]})
kTable = pd.DataFrame(rows)
print(kTable.round(2).to_string(index=False))
# keep the largest K where no depot has to ship
K = max(r['K'] for r in rows if r['Depots shipped'] == 0)
print(f'\nChosen K = {K} (largest K for which no depot has to ship)')

 K    Cluster sizes  Depots not on cluster routes  Routes  Depots shipped  Total cost $
 2         [11, 25]                            17       5               0        835.28
 3       [9, 9, 18]                             8       5               0        846.50
 4    [4, 8, 8, 16]                             5       5               1       1023.33
 5 [4, 5, 6, 7, 14]                             2       5               2       1111.48

Chosen K = 3 (largest K for which no depot has to ship)


### Build the plan

In [11]:
routes, shipDepots, firstLeftover = build_plan(K)
clusters = assign_clusters(K)
for center, members in clusters.items():
    print(f'Cluster centered on {center}: {len(members)} depots')
print(f'\n{firstLeftover} depots did not fit on a cluster route; {len(routes) - K} repair route(s) added; {len(shipDepots)} depots ship directly')

Cluster centered on Andrew County: 9 depots
Cluster centered on Jackson County Altus: 18 depots
Cluster centered on Washington Regional Medical Center: 9 depots

8 depots did not fit on a cluster route; 2 repair route(s) added; 0 depots ship directly


### Plan and cost

In [12]:
days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']

dayRows, stopRows = [], []
for k, route in enumerate(routes):
    # add up the minutes and miles of every leg of this route (including leaving and returning to the bank)
    legs = list(zip(route[:-1], route[1:]))
    T = sum(time[a, b] for a, b in legs)
    D = sum(dist[a, b] for a, b in legs)
    dayRows.append({'Day': days[k], 'Depots visited': len(route) - 2, 'Driving minutes': T, 'Driving hours': T / 60,
                    'Miles': D, 'Labor $': laborPerMin * T, 'Fuel $': fuelPerMile * D})
    cum = 0.0
    for s, (a, b) in enumerate(legs, start=1):
        cum += time[a, b]
        stopRows.append({'Day': days[k], 'Leg': s, 'From': a, 'To': b, 'Minutes': time[a, b],
                         'Arrive (min)': cum, 'Miles': dist[a, b]})

dayTable = pd.DataFrame(dayRows)
stopTable = pd.DataFrame(stopRows)
shipTable = pd.DataFrame([{'Depot': i, 'Miles to bank': dist[i, BANK], 'Shipping $': SHIP_RATE * dist[i, BANK]}
                          for i in shipDepots])

# totals for the whole week
labor = dayTable['Labor $'].sum()
fuel = dayTable['Fuel $'].sum()
shipping = shipTable['Shipping $'].sum() if len(shipTable) else 0.0
total = labor + fuel + shipping

print(dayTable.round(2).to_string(index=False))
print()
print(f'Total driving time : {dayTable["Driving minutes"].sum():,.1f} min ({dayTable["Driving hours"].sum():.2f} h)')
print(f'Total truck miles  : {dayTable["Miles"].sum():,.1f}')
print(f'Labor              : ${labor:,.2f}')
print(f'Fuel               : ${fuel:,.2f}')
print(f'Direct shipping    : ${shipping:,.2f}  ({len(shipDepots)} depots)')
print(f'TOTAL WEEKLY COST  : ${total:,.2f}')

      Day  Depots visited  Driving minutes  Driving hours  Miles  Labor $  Fuel $
   Monday               5           717.45          11.96 837.53    86.69  107.68
  Tuesday              14           643.32          10.72 627.32    77.73   80.66
Wednesday               9           647.87          10.80 681.27    78.28   87.59
 Thursday               4           714.50          11.91 790.07    86.34  101.58
   Friday               4           548.47           9.14 572.98    66.27   73.67

Total driving time : 3,271.6 min (54.53 h)
Total truck miles  : 3,509.2
Labor              : $395.32
Fuel               : $451.18
Direct shipping    : $0.00  (0 depots)
TOTAL WEEKLY COST  : $846.50


### Route detail (ordered stops)

In [13]:
for k, route in enumerate(routes):
    print(f'{days[k]}: ' + ' → '.join(route))
    print()
print('Ship directly:', shipDepots if shipDepots else 'none')

Monday: Oklahoma Mothers’ Milk Bank → Andrew County → KUMC - University of Kansas Health System → Shawnee Mission MC → KUMC - Kansas Health Systems – St. Francis Campus → Kansas Birth Justice Society → Oklahoma Mothers’ Milk Bank

Tuesday: Oklahoma Mothers’ Milk Bank → Alliance Health Woodward → Beckham County → Custer County → Canadian Valley → Integris Baptist → OU Children’s Hospital → Integris Southwest → Variety Care- Lafayette → Cleveland County Moore → Edmond Hope Center → St. Anthony Shawnee Hospital → Pontotoc County → South Central Medical Center → Grady County → Oklahoma Mothers’ Milk Bank

Wednesday: Oklahoma Mothers’ Milk Bank → Washington Regional Medical Center → YWCA East → YWCA South → Tulsa Hillcrest Hospital → OBI → St. John Owasso → Jane Phillips Medical Center → McIntosh County → McAlester Regional Health → Oklahoma Mothers’ Milk Bank

Thursday: Oklahoma Mothers’ Milk Bank → Flint Hills- Riley County Health Department → Salina Regional → Southeast KS Health Departm

### Independent feasibility check
Re-checks every operating rule from the case against the plan above, using the original data.

In [14]:
visited = [d for r in routes for d in r[1:-1]]
assert len(routes) <= MAX_ROUTES, 'more than 5 routes'
assert all(r[0] == BANK and r[-1] == BANK for r in routes), 'every route must start and end at the bank'
assert sorted(visited + shipDepots) == sorted(DEPOTS), 'every depot must be served exactly once'
assert len(visited) == len(set(visited)), 'a depot is visited twice'
assert not (set(visited) & set(shipDepots)), 'a depot both ships and is picked up'
for k, r in enumerate(routes):
    T = sum(time[a, b] for a, b in zip(r[:-1], r[1:]))
    assert T <= MAX_MINUTES + 1e-9, f'{days[k]} exceeds 720 minutes'
assert abs(total - cost_of(routes, shipDepots)[5]) < 1e-9, 'cost components do not add up'
print('All operating rules satisfied.')

All operating rules satisfied.
